In [3]:
import os
import pickle
import pandas as pd
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# if an event is cell type-specific, and it has a exon_diff_boundary or exon_diff_junction transcript, I need to know if:

# (1) the transcript was emitted and 
# (2) it is also cell type-specific.

# this is important because it means that any overlapping domains of interest are not necessarily cell type-specific

# Load in event-protein mapping and event-interproscan results mapping

In [4]:
with open('data/event_interproscan_map.pkl', "rb") as f:
    event_interproscan_map = pickle.load(f)
    
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [5]:
event_interproscan_map

defaultdict(dict,
            {'ENSG00000187634_ProteinCoding_1': {'inclusion':   protein_accession  sequence_length analysis   signature_description  start  \
              0   ENST00000616016              844  Phobius  Non cytoplasmic domain     27   
              
                 stop interpro_description  aa_start  aa_end  exon_cds_start  exon_cds_end  \
              0   844                    -       263     280          931039        931089   
              
                 frame_preserving  clean_start  clean_end  
              0              True        False      False  ,
              'synthetic_skip':                                 protein_accession  sequence_length  \
              0  ENSG00000187634_ProteinCoding_1_synthetic_skip              827   
              1  ENSG00000187634_ProteinCoding_1_synthetic_skip              827   
              
                    analysis          signature_description  start  stop  \
              0  MobiDB-lite  Consensus disord

# Load in significant events for each cell type

In [12]:
signif_events_by_ct = defaultdict(dict)

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        signif_events_by_ct[ctype] = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)

# Plot

In [ ]:
"""
plot_domain_structure.py

Visualizes protein domain structure for a splicing event and its isoforms.

Layout per isoform:
  - Shared x-axis (amino acid position, anchored to inclusion protein)
  - Grey bar = full protein
  - Colored highlight = cassette exon region
  - Domain blocks above the bar (overlapping the exon region, from event_interproscan_map)
  - Domain blocks below the bar (full protein context, from ipr_grouped)
  - Dashed vertical lines at exon aa_start / aa_end
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, FancyBboxPatch
import matplotlib.gridspec as gridspec


ANALYSIS_COLORS = {
    'Pfam':             '#4E79A7',
    'SMART':            '#F28E2B',
    'SUPERFAMILY':      '#E15759',
    'CDD':              '#76B7B2',
    'PROSITE profiles': '#59A14F',
    'PROSITE patterns': '#EDC948',
    'Phobius':          '#B07AA1',
    'DeepTMHMM':        '#FF9DA7',
    'CATH-Gene3D':      '#9C755F',
    'CATH-FunFam':      '#BAB0AC',
    'SignalP':          '#D37295',
    'COILS':            '#A0CBE8',
    'PIRSR':            '#8CD17D',
}
DEFAULT_COLOR = '#bbbbbb'


def _acolor(analysis):
    return ANALYSIS_COLORS.get(analysis, DEFAULT_COLOR)


def _pack_tracks(df, max_tracks=4):
    """Greedy interval packing. Returns dict: row_idx -> track_idx."""
    if df is None or df.empty:
        return {}
    df = df.sort_values('start').reset_index(drop=True)
    tracks = []
    row_to_track = {}
    for i, row in df.iterrows():
        placed = False
        for ti, track in enumerate(tracks):
            if track[-1] < row['start']:
                track.append(row['stop'])
                row_to_track[i] = ti
                placed = True
                break
        if not placed and len(tracks) < max_tracks:
            tracks.append([row['stop']])
            row_to_track[i] = len(tracks) - 1
    return row_to_track


def _draw_domain_tracks(ax, df, y_base, direction, track_h=0.22, gap=0.08,
                         max_tracks=4, alpha=0.85):
    """Draw domain blocks above (direction=1) or below (direction=-1) y_base."""
    if df is None or df.empty:
        return
    df = df.sort_values('start').reset_index(drop=True)
    row_to_track = _pack_tracks(df, max_tracks=max_tracks)
    for i, row in df.iterrows():
        ti = row_to_track.get(i)
        if ti is None:
            continue
        y = y_base + direction * (gap + ti * (track_h + 0.04) + track_h / 2)
        color = _acolor(row.get('analysis', ''))
        s, e = row['start'], row['stop']
        w = max(e - s, 0.5)
        ax.add_patch(FancyBboxPatch(
            (s, y - track_h / 2), w, track_h,
            boxstyle='round,pad=0.3',
            facecolor=color, edgecolor='none',
            alpha=alpha, zorder=4
        ))
        label = str(row.get('signature_description') or row.get('interpro_description') or '')
        if label and w > 8:
            ax.text(s + w / 2, y, label, ha='center', va='center',
                    fontsize=4.2, color='white', fontweight='bold',
                    zorder=5, clip_on=True)


def plot_event_domains(ev, event_protein_map, event_interproscan_map,
                       ipr_grouped, signif_info=None,
                       figsize=None, save_path=None, dpi=150):
    """
    Parameters
    ----------
    ev                     : event ID string
    event_protein_map      : built by build_event_protein_map
    event_interproscan_map : built by the interproscan mapping loop
    ipr_grouped            : {protein_accession: DataFrame} of all ipr results
    signif_info            : optional dict for gene name in title
    figsize / save_path / dpi : standard matplotlib args
    """
    rec     = event_protein_map.get(ev)
    ipr_rec = event_interproscan_map.get(ev, {})

    if rec is None:
        print(f"{ev} not in event_protein_map")
        return None

    rec_incl  = rec['inclusion']
    aa_start  = rec_incl['aa_start']
    aa_end    = rec_incl['aa_end']
    incl_t    = rec_incl['transcript_id']
    incl_full = ipr_grouped.get(incl_t)
    incl_len  = (int(incl_full['sequence_length'].iloc[0])
                 if incl_full is not None and not incl_full.empty
                 else aa_end + 50)

    # ── assemble isoform rows ──────────────────────────────────────────────
    buckets = []

    buckets.append({
        'label':       f"inclusion\n{incl_t}",
        'transcript':  incl_t,
        'aa_start':    aa_start,
        'aa_end':      aa_end,
        'exon_color':  '#F28E2B',
        'targeted':    ipr_rec.get('inclusion'),
        'full':        incl_full,
        'seq_len':     incl_len,
        'truncation':  None,
    })

    if rec.get('real_skip'):
        t = rec['real_skip']
        full = ipr_grouped.get(t)
        slen = (int(full['sequence_length'].iloc[0])
                if full is not None and not full.empty else incl_len)
        buckets.append({
            'label':      f"real skip\n{t}",
            'transcript': t,
            'aa_start':   None, 'aa_end': None,
            'exon_color': '#4E79A7',
            'targeted':   ipr_rec.get('real_skip'),
            'full':       full,
            'seq_len':    slen,
            'truncation': aa_start,
        })

    if rec.get('synthetic_skip'):
        t = rec['synthetic_skip']
        full = ipr_grouped.get(t)
        slen = (int(full['sequence_length'].iloc[0])
                if full is not None and not full.empty else incl_len)
        buckets.append({
            'label':      f"synthetic skip\n{t}",
            'transcript': t,
            'aa_start':   None, 'aa_end': None,
            'exon_color': '#76B7B2',
            'targeted':   ipr_rec.get('synthetic_skip'),
            'full':       full,
            'seq_len':    slen,
            'truncation': aa_start,
        })

    bnd_df = ipr_rec.get('exon_diff_boundary_siblings')
    for sib in rec.get('exon_diff_boundary_siblings', []):
        t = sib['transcript_id']
        full = ipr_grouped.get(t)
        slen = (int(full['sequence_length'].iloc[0])
                if full is not None and not full.empty else incl_len)
        targeted = (bnd_df[bnd_df['protein_accession'] == t]
                    if bnd_df is not None else None)
        buckets.append({
            'label':      f"boundary sibling\n{t}",
            'transcript': t,
            'aa_start':   sib['aa_start'],
            'aa_end':     sib['aa_end'],
            'exon_color': '#59A14F',
            'targeted':   targeted,
            'full':       full,
            'seq_len':    slen,
            'truncation': None,
        })

    jxn_df = ipr_rec.get('exon_diff_junction_siblings')
    for sib in rec.get('exon_diff_junction_siblings', []):
        t = sib['transcript_id']
        full = ipr_grouped.get(t)
        slen = (int(full['sequence_length'].iloc[0])
                if full is not None and not full.empty else incl_len)
        targeted = (jxn_df[jxn_df['protein_accession'] == t]
                    if jxn_df is not None else None)
        buckets.append({
            'label':      f"junction sibling\n{t}",
            'transcript': t,
            'aa_start':   sib['aa_start'],
            'aa_end':     sib['aa_end'],
            'exon_color': '#B07AA1',
            'targeted':   targeted,
            'full':       full,
            'seq_len':    slen,
            'truncation': None,
        })

    if not buckets:
        print(f"No isoforms for {ev}")
        return None

    # ── shared x range ────────────────────────────────────────────────────
    x_max = max(b['seq_len'] for b in buckets)
    x_min = -x_max * 0.18

    # ── figure layout ─────────────────────────────────────────────────────
    n   = len(buckets)
    fw  = figsize[0] if figsize else 13
    fh  = figsize[1] if figsize else max(3.5, n * 2.6)
    fig, axes = plt.subplots(n, 1, figsize=(fw, fh), sharex=True)
    if n == 1:
        axes = [axes]

    gene_name = (signif_info or {}).get(ev, {}).get("gene_name")
    if not gene_name:
        gene_name = rec_incl.get("gene", "")

    cds_start = rec_incl.get("exon_cds_start", "")
    cds_end   = rec_incl.get("exon_cds_end", "")
    chrom     = rec_incl.get("chrom", "")
    coord_str = f"{chrom}:{cds_start}-{cds_end}" if chrom else f"{cds_start}-{cds_end}"

    title_parts = [ev]
    if gene_name and gene_name != ev:
        title_parts.append(gene_name)
    title_parts += [f"exon aa {aa_start}–{aa_end}", coord_str]
    fig.suptitle("  |  ".join(title_parts), fontsize=8, y=1.01)

    y_bar   = 0.0    # protein bar centre
    y_above = 0.15   # baseline for domain tracks above bar
    y_below = -0.15  # baseline for domain tracks below bar

    for ax, b in zip(axes, buckets):

        # ── protein bar ───────────────────────────────────────────────────
        ax.add_patch(Rectangle((0, y_bar - 0.06), b['seq_len'], 0.12,
                                facecolor='#dddddd', edgecolor='none', zorder=1))

        # ── cassette exon highlight ───────────────────────────────────────
        if b['aa_start'] is not None and b['aa_end'] is not None:
            ew = b['aa_end'] - b['aa_start']
            ax.add_patch(Rectangle(
                (b['aa_start'], y_bar - 0.12), max(ew, 1), 0.24,
                facecolor=b['exon_color'], edgecolor='#555555',
                linewidth=0.6, alpha=0.85, zorder=2
            ))
            # boundary tick marks
            for x in (b['aa_start'], b['aa_end']):
                ax.axvline(x, color='#444444', lw=0.9,
                            linestyle='--', alpha=0.7, zorder=3)
            # exon label
            mid = (b['aa_start'] + b['aa_end']) / 2
            ax.text(mid, y_bar + 0.17, f"aa {b['aa_start']}–{b['aa_end']}",
                    ha='center', va='bottom', fontsize=4.5, color='#333333', zorder=6)

        # ── truncation marker ─────────────────────────────────────────────
        if b['truncation'] is not None:
            ax.axvline(b['truncation'], color='#E15759', lw=1.2,
                        linestyle=':', alpha=0.85, zorder=3)
            ax.text(b['truncation'] + 1, y_bar + 0.17, f"truncation\naa {b['truncation']}",
                    ha='left', va='bottom', fontsize=4, color='#E15759', zorder=6)

        # ── targeted domain hits ABOVE bar ────────────────────────────────
        _draw_domain_tracks(ax, b['targeted'], y_above,
                             direction=1, track_h=0.22, gap=0.06,
                             max_tracks=4, alpha=0.9)

        # ── full protein context BELOW bar ────────────────────────────────
        _draw_domain_tracks(ax, b['full'], y_below,
                             direction=-1, track_h=0.17, gap=0.06,
                             max_tracks=3, alpha=0.55)

        # ── axes formatting ───────────────────────────────────────────────
        ax.set_xlim(x_min, x_max * 1.03)
        ax.set_ylim(-1.4, 1.6)
        ax.set_yticks([])
        ax.tick_params(axis='x', labelsize=5.5)
        for spine in ('top', 'left', 'right'):
            ax.spines[spine].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)

        # isoform label
        ax.text(x_min * 0.95, y_bar, b['label'],
                ha='right', va='center', fontsize=6,
                color='#333333', linespacing=1.4)

        # divider between rows
        ax.axhline(-1.35, color='#eeeeee', lw=0.6, zorder=0)

    axes[-1].set_xlabel('amino acid position', fontsize=6)



    # ── legend ────────────────────────────────────────────────────────────
    seen_analyses = set()
    for b in buckets:
        for df in [b['targeted'], b['full']]:
            if df is not None and not (hasattr(df, 'empty') and df.empty):
                seen_analyses.update(df.get('analysis', pd.Series()).unique())
    legend_handles = [
        mpatches.Patch(facecolor=_acolor(a), label=a, alpha=0.85)
        for a in sorted(seen_analyses) if a in ANALYSIS_COLORS
    ]
    if legend_handles:
        fig.legend(handles=legend_handles,
                   loc='lower center',
                   ncol=min(len(legend_handles), 7),
                   fontsize=5.5, frameon=False,
                   bbox_to_anchor=(0.5, -0.03))

    plt.tight_layout(pad=1.0)
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"saved -> {save_path}")
    else:
        plt.show()
    return fig

In [ ]:
ev = 'ENSG00000124333_ProteinCoding_1'
plot_event_domains(ev, event_protein_map, event_interproscan_map, ipr_grouped, signif_info=signif_info)